# Week 2 Mini-Assignment: Texas last statements by Alissa Rivero

**Dataset:** [Last Statements of Executed Offenders](https://www.kaggle.com/datasets/ranjithkumarraik/last-words-of-death-row-inmates) (TDCJ / Kaggle)

- **PreviousCrime = 0** (n = 233) = no prior criminal record
- **PreviousCrime = 1** (n = 276) = prior criminal record
- **PreviousCrime missing** (n = 36) = dropped from the models
- **LastStatement** missing or declined (n = 114) = scored as zero words / zero theme rates

The two original models ask whether a prior record is associated with apology/remorse language and with religious language. A second set of models asks which **last-word themes** show up and whether **demographic categories** (race, age, education, prior record) predict those theme rates.

## Dataset Abstract

The Texas Department of Criminal Justice publishes the last statements of people executed in Texas, together with age, race, county of conviction, education, prior-crime history, and victim counts. This table is a Kaggle export of that archive (545 rows). Statements are short and uneven: some people speak at length, some give a sentence, and about one in five have no recorded statement.

Because longer statements can mention more words of every kind, theme scores are **rates per 100 words** (the same idea as relative abundance in the microbiome assignment). Keyword lists are exact words or clear prefixes. Neutral verbs such as *ask* and *tell* are not treated as themes.

## Research Questions

1. Is a **prior criminal record** associated with more or less **apology / remorse** language?
2. Is a **prior criminal record** associated with more or less **religious** language?
3. What **themes** appear in last words — remorse, gratitude and love, family, and religion — and are those rates predicted by **demographic categories**?


## Load and inspect the dataset

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

DATA_PATH = Path("Texas Last Statement - CSV.csv")

# Remorse / apology. Short function words (ask, tell, say) are not themes.
REMORSE_EXACT = {
    "sorry", "sorrow", "apology", "apologize", "apologise", "apologized",
    "apologised", "apologies", "forgive", "forgave", "forgiven", "forgiveness",
    "forgiving", "remorse", "remorseful", "regret", "regrets", "regretted",
    "regretting", "repent", "repents", "repented", "repentance",
}
REMORSE_PREFIXES = ("apolog", "forgiv", "remorse", "regret", "repent")

GRATITUDE_LOVE_EXACT = {
    "thank", "thanks", "thankful", "thankfully", "gratitude", "grateful",
    "appreciate", "appreciated", "appreciation", "love", "loved", "loves",
    "loving", "lovin",
}
GRATITUDE_LOVE_PREFIXES = ("thank", "gratitude", "grateful", "appreciate")

FAMILY_EXACT = {
    "family", "families", "mom", "moms", "mommy", "mama", "mother", "mothers",
    "mum", "dad", "dads", "daddy", "father", "fathers", "papa", "parent",
    "parents", "wife", "wives", "husband", "husbands", "son", "sons",
    "daughter", "daughters", "kid", "kids", "child", "children", "brother",
    "brothers", "bro", "sister", "sisters", "sis", "sibling", "siblings",
    "grandma", "grandmother", "grandpa", "grandfather", "grandparent",
    "grandparents", "aunt", "aunts", "uncle", "uncles", "nephew", "nephews",
    "niece", "nieces", "cousin", "cousins", "friend", "friends",
    "friendship", "baby", "babies",
}
FAMILY_PREFIXES = (
    "mother", "father", "daughter", "brother", "sister", "grandma",
    "grandpa", "grandparent", "friend",
)

RELIGION_EXACT = {
    "god", "gods", "godly", "jesus", "christ", "christian", "christianity",
    "lord", "lords", "heaven", "heavenly", "allah", "pray", "prayer",
    "prayers", "praying", "prayed", "bible", "biblical", "amen", "holy",
    "church", "faith", "bless", "blessed", "blessing", "blessings",
}
RELIGION_PREFIXES = ("jesus", "christ", "heaven", "pray", "prayer", "bible", "bless")


def tokenize(text):
    words = []
    current = []
    for ch in str(text).lower():
        if ch.isalpha() or ch == "'":
            current.append(ch)
        elif current:
            words.append("".join(current))
            current = []
    if current:
        words.append("".join(current))
    return words


def _is_hit(word, exact, prefixes):
    if word in exact:
        return True
    return any(word.startswith(prefix) for prefix in prefixes)


def theme_hits(words, exact, prefixes):
    return sum(_is_hit(word, exact, prefixes) for word in words)


def is_declined(text):
    if pd.isna(text):
        return True
    low = str(text).strip().lower()
    return low in {"none", "", "nan"} or "declined" in low or "no last statement" in low


def add_text_scores(frame):
    out = frame.copy()
    tokens = out["LastStatement"].map(tokenize)
    out["word_count"] = tokens.map(len)
    out["declined"] = out["LastStatement"].map(is_declined)
    out.loc[out["declined"], "word_count"] = 0
    remorse = tokens.map(lambda words: theme_hits(words, REMORSE_EXACT, REMORSE_PREFIXES))
    gratitude = tokens.map(lambda words: theme_hits(words, GRATITUDE_LOVE_EXACT, GRATITUDE_LOVE_PREFIXES))
    family = tokens.map(lambda words: theme_hits(words, FAMILY_EXACT, FAMILY_PREFIXES))
    religion = tokens.map(lambda words: theme_hits(words, RELIGION_EXACT, RELIGION_PREFIXES))
    out["remorse_hits"] = remorse
    out["gratitude_love_hits"] = gratitude
    out["family_hits"] = family
    out["religion_hits"] = religion
    out["apology_hits"] = remorse
    denom = out["word_count"].astype(float).where(out["word_count"] > 0)
    out["remorse_rate"] = (remorse.astype(float) / denom * 100).fillna(0.0)
    out["apology_rate"] = out["remorse_rate"]
    out["gratitude_love_rate"] = (gratitude.astype(float) / denom * 100).fillna(0.0)
    out["family_rate"] = (family.astype(float) / denom * 100).fillna(0.0)
    out["religion_rate"] = (religion.astype(float) / denom * 100).fillna(0.0)
    return out


THEME_RATES = [
    "apology_rate", "remorse_rate", "gratitude_love_rate",
    "family_rate", "religion_rate",
]

# Load and inspect: first rows, then dtypes / summary / missing values.
# The Kaggle export is Latin-1, not UTF-8. NativeCounty has a trailing space.
df = pd.read_csv(DATA_PATH, encoding="latin-1")
df.columns = df.columns.str.strip()
df.head()


In [ ]:
# dtypes, non-null counts, and numeric summary
df.info()
df.describe()


In [ ]:
# Missing-value counts. LastStatement blanks and "NA" in numeric fields are both missing.
missing = df.apply(
    lambda col: col.isna().sum() + (col.astype(str).str.strip().isin(["", "NA"])).sum()
)
missing


## Split groups

In [ ]:
for col in (
    "Age", "AgeWhenReceived", "EducationLevel", "PreviousCrime",
    "Codefendants", "NumberVictim",
):
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = add_text_scores(df)

print(f"Rows: {len(df)}")
print(f"Declined / no statement: {int(df['declined'].sum())}")
print("PreviousCrime counts (including NA):")
print(df["PreviousCrime"].value_counts(dropna=False).sort_index())

no_prior = df[df["PreviousCrime"] == 0]
prior = df[df["PreviousCrime"] == 1]
print(f"No prior record: {len(no_prior)}")
print(f"Prior record: {len(prior)}")


## `groupby()` summary statistics

`PreviousCrime` 0 vs 1 is the grouping variable. Theme rates are hits per 100 words so a long statement is not automatically a high score.


In [ ]:
labeled = df.dropna(subset=["PreviousCrime"]).copy()
labeled["group"] = labeled["PreviousCrime"].map({0.0: "No prior", 1.0: "Prior"})
labeled.groupby("group")[
    ["word_count", "Age", "EducationLevel", "apology_rate", "religion_rate",
     "gratitude_love_rate", "family_rate"]
].agg(["mean", "count", "std"])


In [ ]:
df.groupby("Race")[
    ["word_count", "Age", "EducationLevel", "remorse_rate",
     "gratitude_love_rate", "family_rate", "religion_rate"]
].agg(["mean", "count", "std"])


## Side-by-side comparison

In [ ]:
comparison_metrics = [
    "word_count", "apology_rate", "religion_rate",
    "gratitude_love_rate", "family_rate",
]
comparison = labeled.groupby(["Race", "group"])[comparison_metrics].mean().unstack()
comparison


In [ ]:
for metric in comparison_metrics:
    denom = comparison[(metric, "No prior")].replace(0, pd.NA)
    comparison[(metric, "percent_diff")] = (
        (comparison[(metric, "Prior")] - comparison[(metric, "No prior")]) / denom * 100
    )
print(comparison.xs("percent_diff", axis=1, level=1))

higher_apology = comparison[("apology_rate", "percent_diff")] > 0
print("\nHigher apology rate with a prior record")
print(comparison.loc[higher_apology, ("apology_rate", "percent_diff")])
print("\nLower apology rate with a prior record")
print(comparison.loc[~higher_apology, ("apology_rate", "percent_diff")])


## Composite theme scores

Four last-word themes, counted as **hits per 100 words**:

- **Remorse** (also called apology in the original models) — *sorry, apologize, forgive, remorse, regret*
- **Gratitude and love** (one theme) — *thank, grateful, love*
- **Family** (kept separate) — *mom, dad, kids, brother, sister, wife, family, friends*, and similar kin words
- **Religion** — *god, jesus, christ, lord, heaven, pray, amen*

Neutral words such as *ask* and *tell* are not included. Short stems are matched as whole words so *son* does not count *person* and *god* does not count *goodbye*.


In [ ]:
model_df = labeled[
    [
        "PreviousCrime", "apology_rate", "religion_rate", "remorse_rate",
        "gratitude_love_rate", "family_rate", "group", "declined",
        "Age", "EducationLevel", "Race",
    ]
].copy()
model_df["prior_crime"] = model_df["PreviousCrime"].astype(int)
for col in THEME_RATES:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce").astype(float)
model_df.groupby("prior_crime")[
    ["apology_rate", "religion_rate", "gratitude_love_rate", "family_rate"]
].agg(["mean", "count", "std"])


## Prediction models

`apology_rate ~ prior_crime` and `religion_rate ~ prior_crime` (prior_crime = 1 for a prior record, 0 for none). These are the two original models.


In [ ]:
X = sm.add_constant(model_df["prior_crime"].astype(float))
apology_model = sm.OLS(model_df["apology_rate"].astype(float), X).fit()
apology_model.summary()


In [ ]:
religion_model = sm.OLS(model_df["religion_rate"].astype(float), X).fit()
religion_model.summary()


### Interpretation: apology / remorse by prior crime

**Question 1: Is a prior criminal record associated with more or less apology / remorse language?**
**No clear difference** in this sample.

- **const = 1.07:** people with no prior record use about 1.07 remorse words per 100 words.
- **prior_crime = −0.003:** people with a prior record are essentially the same (about 1.06 per 100 words). The 95% CI is −0.34 to 0.33 and **crosses zero**.
- **p = 0.99** and **R² ≈ 0:** prior crime does not predict the remorse score.


### Interpretation: religion by prior crime

**Question 2: Is a prior criminal record associated with more or less religious language?**
A **small increase**, not significant at 0.05.

- **const = 1.45:** no-prior statements average about 1.45 religion words per 100 words.
- **prior_crime = +0.50:** prior-record statements are about **0.50 words per 100 higher**. The 95% CI is −0.05 to 1.06 and **crosses zero**.
- **p = 0.072** and **R² = 0.006:** the difference is only suggestive and explains almost none of the variation.


## Themes predicted by demographics

Each theme rate is modeled as `theme ~ prior_crime + Age + EducationLevel + Race`. White is the reference race. The two “Other” rows are dropped. Age and education missing rows are dropped (n = 479).


In [ ]:
demo = model_df.dropna(subset=["Age", "EducationLevel", "Race"]).copy()
demo = demo[demo["Race"].isin(["White", "Black", "Hispanic"])]
demo["Race"] = pd.Categorical(demo["Race"], categories=["White", "Black", "Hispanic"])
X_demo = pd.get_dummies(
    demo[["prior_crime", "Age", "EducationLevel", "Race"]],
    drop_first=True,
).astype(float)
X_demo = sm.add_constant(X_demo)

theme_models = {}
for outcome in ("remorse_rate", "gratitude_love_rate", "family_rate", "religion_rate"):
    theme_models[outcome] = sm.OLS(demo[outcome].astype(float), X_demo).fit()
    print(f"=== {outcome} ~ prior_crime + Age + EducationLevel + Race ===")
    print(theme_models[outcome].summary())
    print()


### Interpretation: themes and demographics

The four themes that actually appear in the statements are remorse, gratitude/love, family, and religion. Function words such as *ask* and *tell* were left out.

- **Remorse:** Black speakers use about **0.45 fewer** remorse words per 100 than White speakers (p = 0.022). Prior crime, age, and education are not associated with remorse. The full model is weak (R² = 0.014, F p = 0.24).
- **Gratitude and love:** Hispanic speakers use about **1.29 more** gratitude/love words per 100 than White speakers (p = 0.009). A prior record is associated with a borderline increase (+0.68, p = 0.054). This is the strongest of the four demographic models (R² = 0.029, F p = 0.015).
- **Family:** no demographic coefficient is significant. Family language is common in every group (about 2 hits per 100 words) and is not predicted by race, age, education, or prior crime (R² = 0.015, F p = 0.20).
- **Religion:** Hispanic speakers use about **0.95 more** religion words per 100 than White speakers (p = 0.025). Prior crime is again only suggestive (+0.54, p = 0.077). R² = 0.023.

**Takeaway:** last-word themes are mostly **gratitude/love** and **family**, then religion and remorse. Demographics explain only a small share of the rates. The clearest associations are **Hispanic speakers using more gratitude/love and more religious language**, and **Black speakers using less remorse language**, relative to White speakers. A prior record does not change remorse and only weakly tracks religion and gratitude.


## Visualization

A **boxplot plus strip plot** is the right chart: two (or three) groups and a continuous rate. The box shows the distribution; the points keep every person visible. Each panel has its own y-axis because the theme rates have different typical ranges.


In [ ]:
sns.set_theme(style="ticks", context="notebook")
plot_df = model_df.copy()
plot_df["Apology / remorse"] = plot_df["apology_rate"]
plot_df["Religious language"] = plot_df["religion_rate"]

panels = [
    ("Apology / remorse", {"No prior": "#D7E2B4", "Prior": "#5E7A32"}, "#3F5320"),
    ("Religious language", {"No prior": "#F3D6E0", "Prior": "#B56B86"}, "#7A3F56"),
]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.2), facecolor="#FBF9F6")
fig.patch.set_facecolor("#FBF9F6")
for ax, (col, palette, edge) in zip(axes, panels):
    ax.set_facecolor("#FBF9F6")
    sns.boxplot(
        data=plot_df, x="group", y=col, hue="group",
        hue_order=["No prior", "Prior"], order=["No prior", "Prior"],
        palette=palette, width=0.55, linewidth=1.1, fliersize=0,
        legend=False, ax=ax,
    )
    for patch in ax.patches:
        patch.set_edgecolor(edge)
        patch.set_linewidth(1.1)
        patch.set_alpha(0.95)
    sns.stripplot(
        data=plot_df, x="group", y=col, hue="group",
        hue_order=["No prior", "Prior"], order=["No prior", "Prior"],
        palette=palette, legend=False, jitter=0.18, size=4.5, alpha=0.55,
        linewidth=0.6, edgecolor="white", ax=ax,
    )
    ax.set_title(col, pad=10, color=edge)
    ax.set_xlabel("")
    ax.set_ylabel("Hits per 100 words")
    sns.despine(ax=ax, trim=True)
    ax.spines["left"].set_color("#C9C3B8")
    ax.spines["bottom"].set_color("#C9C3B8")
    ax.yaxis.grid(True, color="#E6E1D8", linewidth=0.8)
    ax.set_axisbelow(True)

fig.suptitle(
    "Last-statement language by prior criminal record",
    y=1.03, fontsize=16, fontweight="semibold", color="#2F2B26",
)
fig.tight_layout()
fig.savefig("apology_religion.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
fig


In [ ]:
theme_plot = demo.copy()
theme_plot["Remorse"] = theme_plot["remorse_rate"]
theme_plot["Gratitude and love"] = theme_plot["gratitude_love_rate"]
theme_plot["Family"] = theme_plot["family_rate"]
theme_plot["Religion"] = theme_plot["religion_rate"]
theme_panels = [
    ("Remorse", "#5E7A32"),
    ("Gratitude and love", "#B56B86"),
    ("Family", "#3D6B8A"),
    ("Religion", "#8A6A2F"),
]
fig2, axes2 = plt.subplots(1, 4, figsize=(14.5, 5.0), facecolor="#FBF9F6")
fig2.patch.set_facecolor("#FBF9F6")
race_order = ["White", "Black", "Hispanic"]
race_palette = {"White": "#E8E0D2", "Black": "#8A7E6A", "Hispanic": "#C4B49A"}
for ax, (col, edge) in zip(axes2, theme_panels):
    ax.set_facecolor("#FBF9F6")
    sns.boxplot(
        data=theme_plot, x="Race", y=col, hue="Race",
        hue_order=race_order, order=race_order, palette=race_palette,
        width=0.6, linewidth=1.1, fliersize=0, legend=False, ax=ax,
    )
    for patch in ax.patches:
        patch.set_edgecolor(edge)
        patch.set_linewidth(1.1)
    sns.stripplot(
        data=theme_plot, x="Race", y=col, hue="Race",
        hue_order=race_order, order=race_order, palette=race_palette,
        legend=False, jitter=0.18, size=3.5, alpha=0.45,
        linewidth=0.5, edgecolor="white", ax=ax,
    )
    ax.set_title(col, pad=10, color=edge)
    ax.set_xlabel("")
    ax.set_ylabel("Hits per 100 words")
    sns.despine(ax=ax, trim=True)
    ax.spines["left"].set_color("#C9C3B8")
    ax.spines["bottom"].set_color("#C9C3B8")
    ax.yaxis.grid(True, color="#E6E1D8", linewidth=0.8)
    ax.set_axisbelow(True)

fig2.suptitle(
    "Last-word themes by race",
    y=1.03, fontsize=16, fontweight="semibold", color="#2F2B26",
)
fig2.tight_layout()
fig2.savefig("last_statement_themes.png", dpi=150, bbox_inches="tight", facecolor=fig2.get_facecolor())
fig2


## Pandas vs Polars (same pipeline)

The analysis above is Pandas. The same load → keyword-rate → group-mean pipeline can be written in Polars. Times below are the mean of 7 runs after one warmup, on this 545-row table.


In [ ]:
def _repeat(fn, n=7, warmup=1):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(n):
        start = perf_counter()
        fn()
        times.append(perf_counter() - start)
    return sum(times) / len(times)


def _score_frame(frame):
    scored = add_text_scores(frame)
    scored["PreviousCrime"] = pd.to_numeric(scored["PreviousCrime"], errors="coerce")
    scored = scored.dropna(subset=["PreviousCrime"])
    scored["group"] = scored["PreviousCrime"].map({0.0: "No prior", 1.0: "Prior"})
    return scored.groupby("group")[
        ["apology_rate", "religion_rate", "gratitude_love_rate", "family_rate"]
    ].mean()


def pandas_pipeline():
    pdf = pd.read_csv(DATA_PATH, encoding="latin-1")
    pdf.columns = pdf.columns.str.strip()
    return _score_frame(pdf)


def polars_pipeline():
    import polars as pl

    pdf = pl.read_csv(
        str(DATA_PATH),
        encoding="utf8-lossy",
        null_values=["NA", ""],
        infer_schema_length=10000,
    )
    rename = {c: c.strip() for c in pdf.columns if c != c.strip()}
    if rename:
        pdf = pdf.rename(rename)

    remorse = "|".join(sorted(REMORSE_EXACT | set(REMORSE_PREFIXES)))
    gratitude = "|".join(sorted(GRATITUDE_LOVE_EXACT | set(GRATITUDE_LOVE_PREFIXES)))
    family = "|".join(sorted(FAMILY_EXACT | set(FAMILY_PREFIXES)))
    religion = "|".join(sorted(RELIGION_EXACT | set(RELIGION_PREFIXES)))
    declined_pat = r"(?i)^(none)?$|declined|no last statement"

    scored = (
        pdf.with_columns(
            pl.col("LastStatement").str.to_lowercase().alias("text"),
            pl.col("PreviousCrime").cast(pl.Float64, strict=False),
        )
        .with_columns(
            pl.col("text").str.count_matches(r"[A-Za-z']+").alias("word_count"),
            (
                pl.col("LastStatement").is_null()
                | pl.col("text").str.contains(declined_pat)
            ).alias("declined"),
            pl.col("text").str.count_matches(rf"(?i)\b(?:{remorse})").alias("apology_hits"),
            pl.col("text").str.count_matches(rf"(?i)\b(?:{gratitude})").alias("gratitude_love_hits"),
            pl.col("text").str.count_matches(rf"(?i)\b(?:{family})").alias("family_hits"),
            pl.col("text").str.count_matches(rf"(?i)\b(?:{religion})").alias("religion_hits"),
        )
        .with_columns(
            pl.when(pl.col("declined")).then(0).otherwise(pl.col("word_count")).alias("word_count")
        )
        .with_columns(
            pl.when(pl.col("word_count") == 0).then(0.0)
            .otherwise(pl.col("apology_hits") / pl.col("word_count") * 100)
            .alias("apology_rate"),
            pl.when(pl.col("word_count") == 0).then(0.0)
            .otherwise(pl.col("gratitude_love_hits") / pl.col("word_count") * 100)
            .alias("gratitude_love_rate"),
            pl.when(pl.col("word_count") == 0).then(0.0)
            .otherwise(pl.col("family_hits") / pl.col("word_count") * 100)
            .alias("family_rate"),
            pl.when(pl.col("word_count") == 0).then(0.0)
            .otherwise(pl.col("religion_hits") / pl.col("word_count") * 100)
            .alias("religion_rate"),
        )
        .filter(pl.col("PreviousCrime").is_in([0.0, 1.0]))
        .with_columns(
            pl.when(pl.col("PreviousCrime") == 0.0)
            .then(pl.lit("No prior"))
            .otherwise(pl.lit("Prior"))
            .alias("group")
        )
    )
    return scored.group_by("group").agg(
        pl.col("apology_rate").mean(),
        pl.col("religion_rate").mean(),
        pl.col("gratitude_love_rate").mean(),
        pl.col("family_rate").mean(),
    )


print("=== Pandas vs Polars (same pipeline on this CSV) ===")
try:
    import polars as pl  # noqa: F401
except ImportError:
    print("Polars is not installed. pip install polars")
else:
    pandas_s = _repeat(pandas_pipeline)
    polars_s = _repeat(polars_pipeline)
    pandas_cmp = pandas_pipeline()
    polars_rows = {row["group"]: row for row in polars_pipeline().to_dicts()}
    diffs = []
    for group, row in pandas_cmp.iterrows():
        other = polars_rows.get(group)
        if other is None:
            continue
        for col in ("apology_rate", "religion_rate", "gratitude_love_rate", "family_rate"):
            if other.get(col) is None:
                continue
            diffs.append(abs(float(row[col]) - float(other[col])))
    max_abs = max(diffs) if diffs else float("nan")
    print(f"Pandas mean time: {pandas_s * 1000:.1f} ms")
    print(f"Polars mean time: {polars_s * 1000:.1f} ms")
    print(f"Speedup (Pandas / Polars): {pandas_s / polars_s:.2f}x")
    print(f"Max abs difference on overlapping groups: {max_abs:.3e}")


On this machine the same pipeline was about **114 ms in Pandas** and **10 ms in Polars** (~**11×**). Group means are close; Polars uses a word-boundary regex over the same lists, so counts can differ slightly from the Python tokenizer. Polars is faster because the engine is compiled Rust — the same idea as the Question 2 notebook. The scientific results above still come from the Pandas / statsmodels models.
